In [1]:
import os
import math
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
def get_id_number(file_completion):
    file_completion = file_completion.replace('.py','')
    idx = file_completion.find('_')
    file_number = file_completion[idx+1:]
    idx2 =  file_number.find('_')
    if idx2 > -1:
        return int(file_number[:idx2])
    else:
        return int(file_number)

def get_label_validation(result):
    if result == "pass":
        return 0
    return 1

def execute_validation(r):
    if "AssertionError" in r:
        return 1
    elif "failed: ," in r:
        return 1
    elif r == "failed: ":
        return 1
    elif "failed:" in r:
        return 0
    else:
        return 1
        
def get_df_per_program(df):
    grouped_df = df.sort_values(['validation'], ascending=[False])
    grouped_df = grouped_df[grouped_df['lines'].notna()]
    result = grouped_df.groupby('completion_id').head(1).reset_index(drop=True)
    return result

def odex_validator_results(validation_path, answers_path, benchmark="DS", split_lines=True):
    df_answers = pd.read_json(answers_path, lines=True)
    df_answers["execute"] = df_answers["result"].apply(execute_validation)
    df_answers["errors"] =  ~( df_answers["execute"] | df_answers["passed"].astype(bool) )
    df = pd.read_json(validation_path, lines=True)  # 'lines=True' for JSON Lines
    df["validation"] = df["passed"].apply(get_label_validation)
    # Get model name from validation file
    match1 = re.search(r"\/(.*?)_pyright_validation\.jsonl", validation_path)
    if match1:
        df["model"] =  match1.group(1)
        df_answers["model"] =  match1.group(1)
    else:
        match2 = re.search(r"\/(.*?)_pyright_validation\.jsonl", validation_path)
        if match2:
             df["model"] =  match2.group(1)
             df_answers["model"] =  match2.group(1)
    if benchmark == "DS":
        df["completion_id"] = df["completion_id"].apply(get_id_number)
        if split_lines:
            df = get_df_per_program(df)
        result = pd.merge(df, df_answers, on=['completion_id','model'], how='left')
    else:
        df_answers["cid"] = df.index
        df["id"] = df["completion_id"].apply(get_id_number)
        result = pd.merge(df, df_answers, on=['cid','id'], how='left')
    return (result["errors"], result["validation"]), result

def flattened_analysis_pyright(json_output, key=["summary","errorCount"]):
    if isinstance(json_output, str):
        j = json.loads(json_output)
        dct = {}
        if "generalDiagnostics" in j:
            for err in j["generalDiagnostics"]:
                if "severity" in err:
                    if err["severity"] == "error":
                        k = f'line {err["range"]["start"]["line"]}, column {err["range"]["start"]["character"]}'
                        #k = f'line {err["range"]["start"]["line"]}'
                        dct[k] = {}
                        if "message" in err:
                            dct[k]["message"] = err["message"]
                        if "code" in err:
                            dct[k]["code"] = err["rule"]
        return dct
    if math.isnan(json_output):
        return json_output

def convert_mypy_dct(err, dct):
    if "severity" in err:
        if err["severity"] == "error":
            k = f'line {str(int(err["line"])-1)}, column {err["column"]}'
            # k = f'line {str(int(err["line"])-1)}'
            dct[k] = {}
            if "message" in err:
                dct[k]["message"] = err["message"]
            if "code" in err:
                dct[k]["code"] = err["code"]
    return dct

def flattened_analysis_mypy(json_output, key=["summary","errorCount"]):
    if isinstance(json_output, str):
        json_output = json_output.split("\n")
        dct = {}
        for f in json_output:
            if f != "":
                try:
                    j = json.loads(f)
                    dct = convert_mypy_dct(j, dct)
                except:
                    j = {
                        "file": f[0:f.find(".py")],
                        "line":0,
                        "column":0,
                        "message": f[f.find(":"):],
                        "hint": "",
                        "code": "",
                        "severity":"error"
                    }
                    dct = convert_mypy_dct(j, dct)
        return dct
    if math.isnan(json_output):
        return json_output
def cosine_similarity_phrases(phrase1: str, phrase2: str) -> float:
    vectorizer = TfidfVectorizer(stop_words=None)
    try:
        vectors = vectorizer.fit_transform([phrase1, phrase2])
        similarity = cosine_similarity(vectors[0], vectors[1])[0][0]
    except:
        similarity = 1
    return similarity

def find_keywords(w):
    matches = re.findall(r'"(\w+)"', w)
    result = ' '.join(f'"{match}"' for match in matches)
    return result
    
def merge_message(p_k, p_v, m_k, m_v):
    similarity = cosine_similarity_phrases(find_keywords(m_v), find_keywords(p_v))
    return similarity > 0.5

def analysis_pyright(row):
    pyright = row["result_static_pyright_unjson"]
    if len(pyright) > 0:
        return 0
    return 1

def analysis_mypy(row):
    mypy = row["result_static_mypy_unjson"]
    if len(mypy) > 0:
        return 0
    return 1

def analysis_union(row):
    pyright = row["result_static_pyright_unjson"]
    mypy = row["result_static_mypy_unjson"]
    if len(pyright) > 0 and not len(mypy) > 0:
        return 0
    elif not len(pyright) > 0 and len(mypy) > 0:
        return 0
    elif len(pyright) > 0 and len(mypy) > 0:
        return 0
    return 1

def analysis_intersect(row):
    pyright = row["result_static_pyright_unjson"]
    mypy = row["result_static_mypy_unjson"]
    if len(pyright) > 0 and len(mypy) > 0:
        return 0
    return 1
    
def merge_static_analysis_union(row):
    pyright = row["result_static_pyright_unjson"]
    mypy = row["result_static_mypy_unjson"]
    dct_simple = {}
    if isinstance(pyright, dict) and isinstance(mypy, dict):
        dct = {}
        for p_k, p_v in pyright.items():
            for m_k, m_v, in mypy.items():
                is_message = merge_message(p_k, p_v["message"], m_k, m_v["message"])
                if is_message:
                    dct[m_v["message"]] = m_k
                else:
                    dct[p_v["message"]] = p_k
        for a_k, a_v in dct.items():
            dct_simple[a_v] = a_k
    elif isinstance(pyright, dict) and not isinstance(mypy, dict):
        for p_k, p_v in pyright.items():
            dct_simple[p_k] = p_v["message"]
    elif not isinstance(pyright, dict) and isinstance(mypy, dict):
        for p_k, p_v in mypy.items():
            dct_simple[p_k] = p_v["message"]
    return dct_simple

def merge_static_analysis_intersection(row):
    pyright = row["result_static_pyright_unjson"]
    mypy = row["result_static_mypy_unjson"]
    dct_simple = {}
    if isinstance(pyright, dict) and isinstance(mypy, dict):
        dct = {}
        for p_k, p_v in pyright.items():
            for m_k, m_v, in mypy.items():
                is_message = merge_message(p_k, p_v["message"], m_k, m_v["message"])
                if is_message:
                    dct[m_v["message"]] = m_k
        for a_k, a_v in dct.items():
            dct_simple[a_v] = a_k
    return dct_simple

def get_lines_to_test(row):
    pyright = row["lines_pyright"]
    mypy = row["lines_mypy"]
    s = ""
    if isinstance(pyright, str) and not isinstance(mypy, str):
        s = pyright
    elif not isinstance(pyright, str) and isinstance(mypy, str):
        s = mypy
    else:
        s = mypy
    return s

def get_execute_message(row):
    pyright = row["result_execute_pyright"]
    mypy = row["result_execute_mypy"]
    s = ""
    if isinstance(pyright, str) and not isinstance(mypy, str):
        s = pyright
    elif not isinstance(pyright, str) and isinstance(mypy, str):
        s = mypy
    else:
        s = mypy
    return s

def get_static_response(dct):
    s = ""
    for k,v in dct.items():
        if isinstance(v, dict):
            s += f'{k} : {v["message"]}\n'
        else:
            s += f'{k} : {v}\n'
    return s

def get_static_code(text):
    if text != "":
        return 0
    return 1

def get_error_type(text):
    # Regex to match and extract 'AssertionError'
    match = re.search(r"type:\s*<class '([\w.]+)'>", text)
    
    if match:
        extracted_error = match.group(1)
        #if extracted_error == "AssertionError":
            #return "passed"
        #else:
        return extracted_error
    else:
        return text

def get_response(response):
    if response[0].startswith('Yes'):
        return 1
    elif response[0].startswith('No'):
        return 0

In [3]:
true = []
labels = []
validation_files = ["results/odexa/pyright/claude-3-opus-20240229-0_pyright_validation.jsonl", 
                "results/odexa/pyright/gpt-3.5-0125_pyright_validation.jsonl", 
              "results/odexa/pyright/gpt-4-0613_pyright_validation.jsonl",
             "results/odexa/pyright/granite-code-latest-0_pyright_validation.jsonl"]

file_names = ["results/odex/answers/claude-3-opus-20240229-0-answers.jsonl", 
              "results/odex/answers/gpt-3.5-turbo-0125-0-answers.jsonl",
              "results/odex/answers/gpt-4-0613-0-answers.jsonl",
              "results/odex/answers/granite-code-latest-0-answers.jsonl"]

true = []
labels = []
pyright_df = pd.DataFrame()
for i in range(0, len(file_names)):
    x, rs = odex_validator_results(validation_files[i], file_names[i], benchmark="odex", split_lines=False)
    true = true + x[0].values.tolist()
    labels = labels + x[1].values.tolist()
    print(validation_files[i])
    x_1 = x[0].values.tolist()
    x_2 = x[1].values.tolist()
    missing_labels = [] #[True for j in range(len(x_1), 1000)]
    true = true + missing_labels
    labels = labels + missing_labels
    pyright_df = pd.concat([pyright_df, rs])
print(confusion_matrix(true, labels))
print(classification_report(true, labels))

results/odexa/pyright/claude-3-opus-20240229-0_pyright_validation.jsonl
results/odexa/pyright/gpt-3.5-0125_pyright_validation.jsonl
results/odexa/pyright/gpt-4-0613_pyright_validation.jsonl
results/odexa/pyright/granite-code-latest-0_pyright_validation.jsonl
[[702 492]
 [394 432]]
              precision    recall  f1-score   support

       False       0.64      0.59      0.61      1194
        True       0.47      0.52      0.49       826

    accuracy                           0.56      2020
   macro avg       0.55      0.56      0.55      2020
weighted avg       0.57      0.56      0.56      2020



In [4]:
true = []
labels = []
validation_files = ["results/odexa/mypy/claude-3-opus-20240229-0_pyright_validation.jsonl", 
                "results/odexa/mypy/gpt-3.5-0125_pyright_validation.jsonl", 
              "results/odexa/mypy/gpt-4-0613_pyright_validation.jsonl",
             "results/odexa/mypy/granite-code-latest-0_pyright_validation.jsonl"]

file_names = ["results/odex/answers/claude-3-opus-20240229-0-answers.jsonl", 
              "results/odex/answers/gpt-3.5-turbo-0125-0-answers.jsonl",
              "results/odex/answers/gpt-4-0613-0-answers.jsonl",
              "results/odex/answers/granite-code-latest-0-answers.jsonl"]

true = []
labels = []
mypy_df = pd.DataFrame()
for i in range(0, len(file_names)):
    x, rs = odex_validator_results(validation_files[i], file_names[i], benchmark="odex", split_lines=False)
    true = true + x[0].values.tolist()
    labels = labels + x[1].values.tolist()
    print(validation_files[i])
    x_1 = x[0].values.tolist()
    x_2 = x[1].values.tolist()
    missing_labels = [] #[True for j in range(len(x_1), 1000)]
    true = true + missing_labels
    labels = labels + missing_labels
    mypy_df = pd.concat([mypy_df, rs])
print(confusion_matrix(true, labels))
print(classification_report(true, labels))

results/odexa/mypy/claude-3-opus-20240229-0_pyright_validation.jsonl
results/odexa/mypy/gpt-3.5-0125_pyright_validation.jsonl
results/odexa/mypy/gpt-4-0613_pyright_validation.jsonl
results/odexa/mypy/granite-code-latest-0_pyright_validation.jsonl
[[812 382]
 [546 280]]
              precision    recall  f1-score   support

       False       0.60      0.68      0.64      1194
        True       0.42      0.34      0.38       826

    accuracy                           0.54      2020
   macro avg       0.51      0.51      0.51      2020
weighted avg       0.53      0.54      0.53      2020



In [5]:
odex_errors_annotated = pd.read_csv("results/classifier/annotated_odex_errors.csv")

In [6]:
mypy_df = mypy_df.rename(columns={'result_x':'result_static', 'result_y':'result_execute'})
pyright_df = pyright_df.rename(columns={'result_x':'result_static', 'result_y':'result_execute'})
detector_msg = pd.merge(pyright_df, mypy_df, on=['completion_id','cid'], how='outer', suffixes=("_pyright", "_mypy"))
detector_msg["result_static_pyright_unjson"] = detector_msg["result_static_pyright"].apply(flattened_analysis_pyright)
detector_msg["result_static_mypy_unjson"] = detector_msg["result_static_mypy"].apply(flattened_analysis_mypy)
detector_msg["static_union"] = detector_msg.apply(merge_static_analysis_union, axis=1)
detector_msg["static_intersection"] = detector_msg.apply(merge_static_analysis_intersection, axis=1)
detector_msg["static_union_str"] = detector_msg["static_union"].apply(get_static_response)
detector_msg["static_intersection_str"] = detector_msg["static_intersection"].apply(get_static_response)
detector_msg["lines"] = detector_msg.apply(get_lines_to_test, axis=1)
detector_msg["pyright"] = detector_msg.apply(analysis_pyright, axis=1)
detector_msg["mypy"] = detector_msg.apply(analysis_mypy, axis=1)
detector_msg["pyright_str"] = detector_msg["result_static_pyright_unjson"].apply(get_static_response)
detector_msg["mypy_str"] = detector_msg["result_static_mypy_unjson"].apply(get_static_response)
detector_msg["union"] = detector_msg.apply(analysis_union, axis=1)
detector_msg["intersect"] = detector_msg.apply(analysis_intersect, axis=1)
detector_msg["execute_message"] = detector_msg.apply(get_execute_message, axis=1)
detector_msg["execute"] = detector_msg["execute_message"].apply(execute_validation)
annotated_total = odex_errors_annotated[["completion_id","cid","error_type","errors"]]
detector_msg = pd.merge(detector_msg, annotated_total, on=['completion_id','cid'], how='outer')
detector_msg["result_static_pyright_unjson"] = detector_msg["result_static_pyright"].apply(flattened_analysis_pyright)
detector_msg["result_static_mypy_unjson"] = detector_msg["result_static_mypy"].apply(flattened_analysis_mypy)

In [7]:
static_summary = detector_msg[['completion_id','cid','model_x_mypy','execute', 'code_pyright', 'lines','pyright_str','mypy_str','pyright','mypy','intersect','error_type','execute_message']]

In [8]:
static_summary['model'] = static_summary['model_x_mypy'].str.replace('odexa/mypy/','')
static_summary['completion_id'] = static_summary['completion_id'].str.replace('.py','')
static_summary['id'] = static_summary['completion_id'].apply(lambda x: x[x.find('_')+1:])

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_26469/417902823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  static_summary['model'] = static_summary['model_x_mypy'].str.replace('odexa/mypy/','')
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_26469/417902823.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  static_summary['completion_id'] = static_summary['completion_id'].str.replace('.py','')
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_26469/417902823.py:3: Se

In [9]:
static_summary = static_summary[['id','cid','model','pyright_str','mypy_str','mypy','pyright','intersect','code_pyright','lines','execute','error_type','execute_message']]

In [14]:
static_summary.groupby(['model'])['id'].count()

model
claude-3-opus-20240229-0    505
gpt-3.5-0125                505
gpt-4-0613                  505
granite-code-latest-0       505
Name: id, dtype: int64

In [140]:
merged_static_grammar

,id,cid,model,pyright_str,mypy_str,mypy,pyright,intersect,code_pyright,lines_x,execute_x,error_type,execute_message_x,grammar_str,grammar_valid,lines_y,execute_y,execute_message_y
0,101677,277,claude-3-opus-20240229-0,,,1.0,1.0,1.0,"[\ndef f_101677(num):\n """"""Отбросить незнач...","\ndef f_101677(num):\n """"""Отбросить незнача...",1.0,NaN,failed:,Input string is valid according to the grammar.\n,True,def f_101677(num):\n \n return str(num)....,1.0,failed:
1,104440,495,claude-3-opus-20240229-0,,,1.0,1.0,1.0,"[\ndef f_104440(df):\n """"""¿Cómo puedo susti...","\ndef f_104440(df):\n """"""¿Cómo puedo sustit...",1.0,NaN,passed,Input string is valid according to the grammar.\n,True,def f_104440(df):\n \n df = df.replace({...,1.0,passed
2,105134,271,claude-3-opus-20240229-0,,,1.0,1.0,1.0,[\nimport os\nfrom django.utils import timezon...,\nimport os\nfrom django.utils import timezone...,0.0,TypeError,failed: f_105134() missing 2 required position...,Input string is valid according to the grammar.\n,True,import os\nfrom django.utils import timezone\n...,0.0,failed: f_105134() missing 2 required position...
3,10607688,226,claude-3-opus-20240229-0,"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at...",0.0,0.0,0.0,[\nfrom datetime import datetime\n\ndef f_1060...,\nfrom datetime import datetime\n\ndef f_10607...,0.0,AttributeError,failed: module 'datetime' has no attribute 'now',Input string is invalid according to the gramm...,False,from datetime import datetime\n\ndef f_1060768...,0.0,failed: module 'datetime' has no attribute 'now'
4,107612,497,claude-3-opus-20240229-0,,,1.0,1.0,1.0,"[\ndef f_107612(file1, file2, file3):\n """"""...","\ndef f_107612(file1, file2, file3):\n """"""¿...",1.0,NaN,failed:,Input string is valid according to the grammar.\n,True,"def f_107612(file1, file2, file3):\n \n ...",1.0,failed:
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016,9760588,85,granite-code-latest-0,,,1.0,1.0,1.0,Here is one implementation. We use regular exp...,"\nimport re\n\ndef f_9760588(myString):\n ""...",0.0,AttributeError,failed: 'tuple' object has no attribute 'replace',Input string is valid according to the grammar.\n,True,import re\n\ndef f_9760588(myString):\n \n ...,0.0,failed: 'tuple' object has no attribute 'replace'
2017,9775297,178,granite-code-latest-0,,,1.0,1.0,1.0,"def f_9775297(a, b):\n """"""append a numpy ar...","\ndef f_9775297(a, b):\n """"""append a numpy ...",0.0,ValueError,failed: could not broadcast input array from s...,Input string is valid according to the grammar.\n,True,"def f_9775297(a, b):\n \n import numpy a...",0.0,failed: could not broadcast input array from s...
2018,9836,348,granite-code-latest-0,,,1.0,1.0,1.0,"import random\n\ndef f_9836(li):\n """"""リスト`l...","\nimport random\n\ndef f_9836(li):\n """"""リスト...",1.0,NaN,passed,Input string is valid according to the grammar.\n,True,import random\n\ndef f_9836(li):\n \n if...,1.0,passed
2019,98725,498,granite-code-latest-0,,,1.0,1.0,1.0,"def f_98725(lst):\n """"""¿Cómo obtener la pos...","\ndef f_98725(lst):\n """"""¿Cómo obtener la p...",0.0,TypeError,failed: '>' not supported between instances of...,Input string is valid according to the grammar.\n,True,def f_98725(lst):\n \n if len(lst) == 0:...,0.0,failed: '>' not supported between instances of...


In [ ]:
model_names = ['claude-3-opus-20240229-0','gpt-3.5-0125','gpt-4-0613','granite-code-latest-0']
col_exp = 'mypy'
imaginary = True
syntax = True
for m in model_names:
    df_m = static_summary[static_summary['model'] == m]
    #df_m['grammar'] = df_m['grammar_valid'].astype(float)
    if syntax:
        df_m = df_m[df_m['error_type'].isin(['SyntaxError'])]
    elif imaginary:
        df_m = df_m[df_m['error_type'].isin(['ImportError','ModuleNotFoundError','AttributeError','TypeError'])]
    print(confusion_matrix(df_m['execute'], df_m[col_exp]))
    print(classification_report(df_m['execute'], df_m[col_exp]))

In [39]:
messages_static_grammar[messages_static_grammar['error_type'] == 'SyntaxError'].groupby('grammar_str')['execute_x'].count()

grammar_str
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                138
Input string is invalid according to the grammar.\nError: Unexpected character '\n' at position 16\n\nInput string:\nimport cv2 as cv\n\nimg = cv.imread('file.png', cv.IMREAD_GRAYSCALE)\n\n\n                                                                                                                                                         

In [32]:
messages_static_grammar

,id,cid,model,error_type,execute_message_x,pyright_str,mypy_str,grammar_str,execute_x,llm_response
0,101677,277,claude-3-opus-20240229-0,NaN,failed:,,,,1.0,"\ndef f_101677(num):\n """"""Отбросить незнача..."
1,104440,495,claude-3-opus-20240229-0,NaN,passed,,,,1.0,"\ndef f_104440(df):\n """"""¿Cómo puedo sustit..."
2,105134,271,claude-3-opus-20240229-0,TypeError,failed: f_105134() missing 2 required position...,,,,0.0,\nimport os\nfrom django.utils import timezone...
3,10607688,226,claude-3-opus-20240229-0,AttributeError,failed: module 'datetime' has no attribute 'now',"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at...",Input string is invalid according to the gramm...,0.0,\nfrom datetime import datetime\n\ndef f_10607...
4,107612,497,claude-3-opus-20240229-0,NaN,failed:,,,,1.0,"\ndef f_107612(file1, file2, file3):\n """"""¿..."
...,...,...,...,...,...,...,...,...,...,...
2016,9760588,85,granite-code-latest-0,AttributeError,failed: 'tuple' object has no attribute 'replace',,,,0.0,Here is one implementation. We use regular exp...
2017,9775297,178,granite-code-latest-0,ValueError,failed: could not broadcast input array from s...,,,,0.0,"def f_9775297(a, b):\n """"""append a numpy ar..."
2018,9836,348,granite-code-latest-0,NaN,passed,,,,1.0,"import random\n\ndef f_9836(li):\n """"""リスト`l..."
2019,98725,498,granite-code-latest-0,TypeError,failed: '>' not supported between instances of...,,,,0.0,"def f_98725(lst):\n """"""¿Cómo obtener la pos..."


In [38]:
model_names = ['claude-3-opus-20240229-0','gpt-3.5-0125','gpt-4-0613','granite-code-latest-0']
col_exp = 'grammar_valid'
imaginary = True
syntax = True
for m in model_names:
    df_m = messages_static_grammar[messages_static_grammar['model'] == m]
    df_m['grammar_valid'] = ~df_m['grammar_str'].apply(lambda x : len(x) > 0)
    #df_m['grammar'] = df_m['grammar_valid'].astype(float)
    if syntax:
        df_m = df_m[df_m['error_type'].isin(['SyntaxError'])]
    elif imaginary:
        df_m = df_m[df_m['error_type'].isin(['ImportError','ModuleNotFoundError','AttributeError','TypeError'])]
    print(confusion_matrix(df_m['execute_x'], df_m[col_exp].astype(int)))
    print(classification_report(df_m['execute_x'], df_m[col_exp].astype(int)))

[[0 1]
 [0 0]]
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       1.0
         1.0       0.00      0.00      0.00       0.0

    accuracy                           0.00       1.0
   macro avg       0.00      0.00      0.00       1.0
weighted avg       0.00      0.00      0.00       1.0

[[ 0 23]
 [ 0  0]]
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00      23.0
         1.0       0.00      0.00      0.00       0.0

    accuracy                           0.00      23.0
   macro avg       0.00      0.00      0.00      23.0
weighted avg       0.00      0.00      0.00      23.0

[[ 0 74]
 [ 0  0]]
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00      74.0
         1.0       0.00      0.00      0.00       0.0

    accuracy                           0.00      74.0
   macro avg       0.00      0.00      0.00      74.0
weighted avg       0.

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_26469/609177364.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_m['grammar_valid'] = ~df_m['grammar_str'].apply(lambda x : len(x) > 0)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to

In [11]:
model_names = ['claude-3-opus-20240229-0','gpt-3.5-0125','gpt-4-0613','granite-code-latest-0']
col_exp = 'mypy'
imaginary = True
syntax = True
for m in model_names:
    df_m = static_summary[static_summary['model'] == m]
    #df_m['grammar'] = df_m['grammar_valid'].astype(float)
    if syntax:
        df_m = df_m[df_m['error_type'].isin(['SyntaxError'])]
    elif imaginary:
        df_m = df_m[df_m['error_type'].isin(['ImportError','ModuleNotFoundError','AttributeError','TypeError'])]
    print(confusion_matrix(df_m['execute'], df_m[col_exp]))
    print(classification_report(df_m['execute'], df_m[col_exp]))

[[1]]
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         1

    accuracy                           1.00         1
   macro avg       1.00      1.00      1.00         1
weighted avg       1.00      1.00      1.00         1

[[20  3]
 [ 0  0]]
              precision    recall  f1-score   support

         0.0       1.00      0.87      0.93        23
         1.0       0.00      0.00      0.00         0

    accuracy                           0.87        23
   macro avg       0.50      0.43      0.47        23
weighted avg       1.00      0.87      0.93        23

[[72  2]
 [ 0  0]]
              precision    recall  f1-score   support

         0.0       1.00      0.97      0.99        74
         1.0       0.00      0.00      0.00         0

    accuracy                           0.97        74
   macro avg       0.50      0.49      0.49        74
weighted avg       1.00      0.97      0.99        74

[[18 26]
 [ 0  0]]
          

/opt/homebrew/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classif

In [19]:
grammar_summary['id'] = grammar_summary['completion_id'].apply(lambda x: x[x.find('_')+1:])
grammar_summary = grammar_summary[['id','cid','model','grammar_str','grammar_valid','lines','execute','execute_message']]

In [20]:
merged_static_grammar = static_summary.merge(grammar_summary, on=['id','cid','model'], how='left')

In [26]:
df_ds = pd.read_csv('ds_for_annotation.csv')

In [27]:
df_ds.columns

Index(['completion_id', 'model', 'library', 'test_cases', 'gold_answer',
       'prompt', 'llm_response', 'error_type', 'execute_message',
       'pyright_str', 'mypy_str', 'grammar_str', 'caught_static_right',
       'caught_static_comment', 'capable_static', 'capable_static_comment',
       'caught_grammar_right', 'caught_grammar_comment', 'capable_grammar',
       'capable_grammar_comment'],
      dtype='object')

In [24]:
merged_static_grammar

,id,cid,model,pyright_str,mypy_str,mypy,pyright,intersect,code_pyright,lines_x,execute_x,error_type,execute_message_x,grammar_str,grammar_valid,lines_y,execute_y,execute_message_y
0,101677,277,claude-3-opus-20240229-0,,,1.0,1.0,1.0,"[\ndef f_101677(num):\n """"""Отбросить незнач...","\ndef f_101677(num):\n """"""Отбросить незнача...",1.0,NaN,failed:,Input string is valid according to the grammar.\n,True,def f_101677(num):\n \n return str(num)....,1.0,failed:
1,104440,495,claude-3-opus-20240229-0,,,1.0,1.0,1.0,"[\ndef f_104440(df):\n """"""¿Cómo puedo susti...","\ndef f_104440(df):\n """"""¿Cómo puedo sustit...",1.0,NaN,passed,Input string is valid according to the grammar.\n,True,def f_104440(df):\n \n df = df.replace({...,1.0,passed
2,105134,271,claude-3-opus-20240229-0,,,1.0,1.0,1.0,[\nimport os\nfrom django.utils import timezon...,\nimport os\nfrom django.utils import timezone...,0.0,TypeError,failed: f_105134() missing 2 required position...,Input string is valid according to the grammar.\n,True,import os\nfrom django.utils import timezone\n...,0.0,failed: f_105134() missing 2 required position...
3,10607688,226,claude-3-opus-20240229-0,"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at...",0.0,0.0,0.0,[\nfrom datetime import datetime\n\ndef f_1060...,\nfrom datetime import datetime\n\ndef f_10607...,0.0,AttributeError,failed: module 'datetime' has no attribute 'now',Input string is invalid according to the gramm...,False,from datetime import datetime\n\ndef f_1060768...,0.0,failed: module 'datetime' has no attribute 'now'
4,107612,497,claude-3-opus-20240229-0,,,1.0,1.0,1.0,"[\ndef f_107612(file1, file2, file3):\n """"""...","\ndef f_107612(file1, file2, file3):\n """"""¿...",1.0,NaN,failed:,Input string is valid according to the grammar.\n,True,"def f_107612(file1, file2, file3):\n \n ...",1.0,failed:
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016,9760588,85,granite-code-latest-0,,,1.0,1.0,1.0,Here is one implementation. We use regular exp...,"\nimport re\n\ndef f_9760588(myString):\n ""...",0.0,AttributeError,failed: 'tuple' object has no attribute 'replace',Input string is valid according to the grammar.\n,True,import re\n\ndef f_9760588(myString):\n \n ...,0.0,failed: 'tuple' object has no attribute 'replace'
2017,9775297,178,granite-code-latest-0,,,1.0,1.0,1.0,"def f_9775297(a, b):\n """"""append a numpy ar...","\ndef f_9775297(a, b):\n """"""append a numpy ...",0.0,ValueError,failed: could not broadcast input array from s...,Input string is valid according to the grammar.\n,True,"def f_9775297(a, b):\n \n import numpy a...",0.0,failed: could not broadcast input array from s...
2018,9836,348,granite-code-latest-0,,,1.0,1.0,1.0,"import random\n\ndef f_9836(li):\n """"""リスト`l...","\nimport random\n\ndef f_9836(li):\n """"""リスト...",1.0,NaN,passed,Input string is valid according to the grammar.\n,True,import random\n\ndef f_9836(li):\n \n if...,1.0,passed
2019,98725,498,granite-code-latest-0,,,1.0,1.0,1.0,"def f_98725(lst):\n """"""¿Cómo obtener la pos...","\ndef f_98725(lst):\n """"""¿Cómo obtener la p...",0.0,TypeError,failed: '>' not supported between instances of...,Input string is valid according to the grammar.\n,True,def f_98725(lst):\n \n if len(lst) == 0:...,0.0,failed: '>' not supported between instances of...


In [21]:
def grammar_detect(row):
    if row['grammar_valid'] == False:
        return row['grammar_str']
    else:
        return ""
    
# Change grammar_str if 'valid' empty, 'invalid' message
# Change execute_message_x to execute_message
# Add to odex_validator_results the original lines of the LLM and the prompt
m_s_grammar = merged_static_grammar[['id', 'cid', 'model', 'error_type', 'execute_message_x','code_pyright','pyright_str', 'mypy_str', 'grammar_str', 'grammar_valid','execute_x']]
m_s_grammar['llm_response'] = m_s_grammar['code_pyright'].apply(lambda x: x[0] if isinstance(x, list) else x)
m_s_grammar['grammar_str'] = m_s_grammar.apply(grammar_detect, axis=1)

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_26469/3037006331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  m_s_grammar['llm_response'] = m_s_grammar['code_pyright'].apply(lambda x: x[0] if isinstance(x, list) else x)
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_26469/3037006331.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  m_s_grammar['grammar_str'] = m_s_grammar.apply(grammar_detect, axis=1)


In [22]:
#m_s_grammar = m_s_grammar.rename(columns={'execute_message_x':'execute_message', 'execute_x':'execute'})
messages_static_grammar = m_s_grammar.drop(columns=['code_pyright','grammar_valid'])

In [23]:
def concat_files(files_lst):
    all_responses = []
    for file in files_lst:
        responses = [json.loads(l) for l in open(file, "r")]
        for response in responses:
            last_word = file.split('/')[-1]
            response["lang"] = last_word[0:2]
            all_responses.append(response)
    return pd.DataFrame(all_responses).rename(columns={'task_id':'id'})

In [24]:
file_lst = ['Data/odex/data/en_test.jsonl','Data/odex/data/es_test.jsonl','Data/odex/data/ja_test.jsonl','Data/odex/data/ru_test.jsonl']
test_file = concat_files(file_lst)
test_file['prompt']= test_file['prompt'] + test_file['intent']
test_file = test_file.rename(columns={'test':'test_cases', 'canonical_solution':'gold_answer'})
test_file['id'] = test_file['id'].astype(str)
test_file = test_file.drop_duplicates(subset="id", keep="first")

In [27]:
messages_static_grammar_error = messages_static_grammar[messages_static_grammar['execute_x']==0.0]

In [ ]:
merged_test = pd.merge(messages_static_grammar_error, test_file[['id','prompt','gold_answer','test_cases','library','lang']], on=['id'])

In [ ]:
merged_test = pd.merge(messages_static_grammar, test_file[['id','prompt','gold_answer','test_cases','library','lang']], on=['id'])

In [32]:
merged_summary = merged_test[['id','cid','model','library','lang','test_cases','gold_answer','prompt','llm_response','error_type','execute_x','execute_message_x','pyright_str','mypy_str','grammar_str']]

In [ ]:
merged_summary.to_csv('

In [117]:
merged_summary_hall = merged_summary[merged_summary['error_type'].isin(['ImportError','ModuleNotFoundError','AttributeError','TypeError'])]

In [120]:
merged_summary_hall.groupby('lang')['llm_response'].count()

lang
en    139
es     34
ja    132
ru     68
Name: llm_response, dtype: int64

In [122]:
#merged_summary_hall.groupby('model', group_keys=False).apply(lambda x: x if len(x) <= 50 else x.sample(n=50)).to_csv('odex_for_annotation.csv',index=False)

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_51656/1376434265.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  merged_summary_hall.groupby('model', group_keys=False).apply(lambda x: x if len(x) <= 50 else x.sample(n=50)).to_csv('odex_for_annotation.csv',index=False)


In [92]:
#grammar_summary.to_csv('odex_grammar_explain.csv', index=False)
#static_summary.to_csv('odex_static_explain.csv', index=False)

In [9]:
total = static_summary[static_summary['execute']==0].groupby(['model_x_mypy','execute','mypy'])['lines'].count().reset_index()
caught = total[total['mypy']==0]
all_errors = static_summary[static_summary['execute']==0].groupby(['model_x_mypy','execute'])['lines'].count().reset_index()
per_errors = pd.merge(caught,all_errors,on=['model_x_mypy','execute'])
per_errors['avg'] = per_errors['lines_x']/per_errors['lines_y']
per_errors

,model_x_mypy,execute,mypy,lines_x,lines_y,avg
0,odexa/mypy/claude-3-opus-20240229-0,0.0,0.0,22,152,0.144737
1,odexa/mypy/gpt-3.5-0125,0.0,0.0,132,199,0.663317
2,odexa/mypy/gpt-4-0613,0.0,0.0,172,212,0.811321
3,odexa/mypy/granite-code-latest-0,0.0,0.0,47,263,0.178707


### False negative

In [138]:
fn = static_summary[static_summary["execute"]==0]
fn_static = fn[fn["intersect"]==1]
fn_static.groupby('model_x_mypy')['execute_message'].count()

model_x_mypy
odexa/mypy/claude-3-opus-20240229-0    130
odexa/mypy/gpt-3.5-0125                 68
odexa/mypy/gpt-4-0613                   41
odexa/mypy/granite-code-latest-0       217
Name: execute_message, dtype: int64

In [145]:
filter_values = ['AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError']
g = fn_static[fn_static['error_type'].isin(filter_values)].groupby(['model_x_mypy','error_type'])['execute_message'].count().reset_index()

In [148]:
result = g.pivot(index='model_x_mypy', columns='error_type', values='execute_message')

In [155]:
g_filter = fn_static[fn_static['error_type'].isin(filter_values)]
g_none = g_filter[g_filter['execute_message'].str.contains("failed: 'NoneType' object")]
g_none.groupby(['model_x_mypy'])['execute_message'].count()

model_x_mypy
odexa/mypy/claude-3-opus-20240229-0    17
odexa/mypy/gpt-3.5-0125                10
odexa/mypy/gpt-4-0613                   8
odexa/mypy/granite-code-latest-0       27
Name: execute_message, dtype: int64

In [156]:
g_none = g_filter[g_filter['execute_message'].str.contains("f_")]
g_none.groupby(['model_x_mypy'])['execute_message'].count()

model_x_mypy
odexa/mypy/claude-3-opus-20240229-0     8
odexa/mypy/gpt-3.5-0125                 4
odexa/mypy/gpt-4-0613                   3
odexa/mypy/granite-code-latest-0       15
Name: execute_message, dtype: int64

In [151]:
fn_static[fn_static['error_type'].isin(filter_values)].to_csv('ifs_fn.csv',index=False)

In [25]:
sampled_fn_static = fn_static.groupby('model_x_mypy', group_keys=False).apply(lambda x: x.sample(n=25))

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_52428/3222669024.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_fn_static = fn_static.groupby('model_x_mypy', group_keys=False).apply(lambda x: x.sample(n=25))


In [27]:
sampled_fn_static.to_csv("static_fn_odex.csv",index=False)

In [53]:
fp_static.groupby('model_x_mypy')['execute_message'].count()

model_x_mypy
odexa/mypy/claude-3-opus-20240229-0      4
odexa/mypy/gpt-3.5-0125                208
odexa/mypy/gpt-4-0613                  232
odexa/mypy/granite-code-latest-0        11
Name: execute_message, dtype: int64

### False positive

In [55]:
fp = static_summary[static_summary["execute"]==1]
fp_static = fp[fp["intersect"]==0.0]

sampled_fp_static = fp_static.groupby('model_x_mypy', group_keys=False).apply(lambda x: x if len(x) <= 25 else x.sample(n=25))
sampled_fp_static

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_52428/3341118218.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_fp_static = fp_static.groupby('model_x_mypy', group_keys=False).apply(lambda x: x if len(x) <= 25 else x.sample(n=25))


,completion_id,cid,model_x_mypy,execute,lines,pyright_str,mypy_str,intersect,error_type,execute_message
17,claude-3-opus-20240229-0_12400256.py,117,odexa/mypy/claude-3-opus-20240229-0,1.0,\nfrom datetime import datetime\n\ndef f_12400...,"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at...",0.0,NaN,failed:
26,claude-3-opus-20240229-0_134564.py,259,odexa/mypy/claude-3-opus-20240229-0,1.0,\nfrom datetime import datetime\n\ndef f_13456...,"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at...",0.0,NaN,passed
326,claude-3-opus-20240229-0_39375.py,374,odexa/mypy/claude-3-opus-20240229-0,1.0,\nfrom datetime import datetime\n\ndef f_39375...,"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at...",0.0,NaN,passed
451,claude-3-opus-20240229-0_698223.py,6,odexa/mypy/claude-3-opus-20240229-0,1.0,\nfrom datetime import datetime\n\ndef f_69822...,"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at...",0.0,NaN,failed:
544,gpt-3.5-0125_14401047.py,150,odexa/mypy/gpt-3.5-0125,1.0,"\n[data[:, i:i+2].mean(axis=1) for i in range(...","line 1, column 1 : ""data"" is not defined\nline...","line 1, column 1 : Name ""data"" is not defined\n",0.0,NaN,failed:
...,...,...,...,...,...,...,...,...,...,...
1779,granite-code-latest-0_3320406.py,164,odexa/mypy/granite-code-latest-0,1.0,\nfrom the `os.path` module in Python. The `os...,"line 1, column 5 : Import ""the"" could not be r...","line -1, column 0 : )\n",0.0,NaN,passed
1822,granite-code-latest-0_3777301.py,105,odexa/mypy/granite-code-latest-0,1.0,"\nimport os\nimport sys\nsys.path.append("".."")...","line 5, column 5 : Import ""libs.subproc"" could...","line -1, column 0 : )\n",0.0,NaN,failed:
1852,granite-code-latest-0_3995.py,465,odexa/mypy/granite-code-latest-0,1.0,"\ndef __init__(self, **kwargs):\n for k...","line 5, column 4 : Unindent amount does not ma...","line -1, column 0 : )\n",0.0,NaN,passed
1878,granite-code-latest-0_41536.py,474,odexa/mypy/granite-code-latest-0,1.0,"\nimport os\nfrom typing import List, Tuple\n\...","line 9, column 29 : Argument of type ""tuple[st...","line 5, column 4 : Need type annotation for ""f...",0.0,NaN,timed out


In [56]:
sampled_fp_static.to_csv("static_fp_odex.csv",index=False)

### Grammar

In [13]:
import pickle
# Load the pipeline
with open('pipeline_model.pkl', 'rb') as f:
    pipeline = pickle.load(f)

In [14]:
true = []
labels = []
validation_files = ["results/grammar/odex/claude-3-opus-20240229-0_grammar_validation.jsonl", 
                "results/grammar/odex/gpt-3.5-turbo-0125-0_grammar_validation.jsonl", 
              "results/grammar/odex/gpt-4-0613-0_grammar_validation.jsonl",
             "results/grammar/odex/granite-code-latest-0_grammar_validation.jsonl"]

file_names = ["results/odex/answers/claude-3-opus-20240229-0-answers.jsonl", 
              "results/odex/answers/gpt-3.5-turbo-0125-0-answers.jsonl",
              "results/odex/answers/gpt-4-0613-0-answers.jsonl",
              "results/odex/answers/granite-code-latest-0-answers.jsonl"]

true = []
labels = []
grammar_df = pd.DataFrame()
for i in range(0, len(file_names)):
    x, rs = odex_validator_results(validation_files[i], file_names[i], benchmark="odex", split_lines=False)
    true = true + x[0].values.tolist()
    labels = labels + x[1].values.tolist()
    print(validation_files[i])
    x_1 = x[0].values.tolist()
    x_2 = x[1].values.tolist()
    missing_labels = [] #[True for j in range(len(x_1), 1000)]
    true = true + missing_labels
    labels = labels + missing_labels
    grammar_df = pd.concat([grammar_df, rs])
print(confusion_matrix(true, labels))
print(classification_report(true, labels))

results/grammar/odex/claude-3-opus-20240229-0_grammar_validation.jsonl
results/grammar/odex/gpt-3.5-turbo-0125-0_grammar_validation.jsonl
results/grammar/odex/gpt-4-0613-0_grammar_validation.jsonl
results/grammar/odex/granite-code-latest-0_grammar_validation.jsonl
[[1103   91]
 [ 736   90]]
              precision    recall  f1-score   support

       False       0.60      0.92      0.73      1194
        True       0.50      0.11      0.18       826

    accuracy                           0.59      2020
   macro avg       0.55      0.52      0.45      2020
weighted avg       0.56      0.59      0.50      2020



In [15]:
grammar_df["error_type"] = pipeline.predict(grammar_df["result_y"])

In [16]:
grammar_df = grammar_df.rename(columns={'result_x':'grammar_str', 'result_y': 'execute_message'})
grammar_df['model'] = grammar_df['completion_id'].str.extract(r'^(.*?)_')
grammar_summary = grammar_df[['completion_id','id','cid','model','execute', 'lines','grammar_str','error_type','execute_message','passed_y']]

In [17]:
grammar_summary = grammar_summary.rename(columns={'result_x':'grammar_str', 'result_y': 'execute_message', 'passed_y': 'grammar'})
grammar_summary = grammar_summary[['completion_id','cid','model','execute', 'lines','grammar_str','error_type','execute_message','grammar']]

In [18]:
grammar_summary['grammar_valid'] = grammar_summary['grammar_str'].str.contains('string is valid')

In [21]:
grammar_summary.groupby(['grammar_valid','execute'])['lines'].count()

grammar_valid  execute
False          0            90
               1            91
True           0           736
               1          1103
Name: lines, dtype: int64

In [16]:
total = grammar_summary[grammar_summary['execute']==0].groupby(['model','execute','grammar_valid'])['lines'].count().reset_index()
caught = total[total['grammar_valid']==0]
all_errors = grammar_summary[grammar_summary['execute']==0].groupby(['model','execute'])['lines'].count().reset_index()
per_errors = pd.merge(caught,all_errors,on=['model','execute'])
per_errors['avg'] = per_errors['lines_x']/per_errors['lines_y']
per_errors

,model,execute,grammar_valid,lines_x,lines_y,avg
0,claude-3-opus-20240229-0,0,False,12,152,0.078947
1,gpt-3.5-turbo-0125-0,0,False,15,199,0.075377
2,gpt-4-0613-0,0,False,23,212,0.108491
3,granite-code-latest-0,0,False,40,263,0.152091


### False Negative

In [104]:
fn = grammar_summary[grammar_summary["execute"]==0]
fn_grammar = fn[fn["grammar_str"].str.contains("valid")]
fn_grammar.groupby('model')['execute_message'].count()

model
claude-3-opus-20240229-0    152
gpt-3.5-turbo-0125-0        199
gpt-4-0613-0                212
granite-code-latest-0       263
Name: execute_message, dtype: int64

In [107]:
sampled_fn_grammar = fn_grammar.groupby('model', group_keys=False).apply(lambda x: x if len(x) <= 25 else x.sample(n=25))
sampled_fn_grammar[['completion_id', 'cid', 'model', 'execute', 'lines', 'error_type','execute_message', 'grammar_str']].to_csv("grammar_fn_odex.csv",index=False)

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_52428/3670505279.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_fn_grammar = fn_grammar.groupby('model', group_keys=False).apply(lambda x: x if len(x) <= 25 else x.sample(n=25))


In [106]:
fn

,completion_id,cid,model,execute,lines,grammar_str,error_type,execute_message,grammar
5,claude-3-opus-20240229-0_15405636,13,claude-3-opus-20240229-0,0,def f_15405636(parser):\n \n parser.add_...,Input string is valid according to the grammar.\n,AttributeError,failed: 'ArgumentParser' object has no attribu...,False
9,claude-3-opus-20240229-0_40133826,16,claude-3-opus-20240229-0,0,def f_40133826(mylist):\n \n with open('...,Input string is valid according to the grammar.\n,AssertionError,failed: unpickling stack underflow,False
10,claude-3-opus-20240229-0_22676,9,claude-3-opus-20240229-0,0,import requests\n\ndef f_22676(url):\n \n ...,Input string is valid according to the grammar.\n,TypeError,failed: object of type 'NoneType' has no len(),False
11,claude-3-opus-20240229-0_26153795,3,claude-3-opus-20240229-0,0,from datetime import datetime\n\ndef f_2615379...,Input string is invalid according to the gramm...,AttributeError,failed: module 'datetime' has no attribute 'st...,False
14,claude-3-opus-20240229-0_22676,10,claude-3-opus-20240229-0,0,import requests\n\ndef f_22676(url):\n \n ...,Input string is valid according to the grammar.\n,TypeError,failed: argument of type 'NoneType' is not ite...,False
...,...,...,...,...,...,...,...,...,...
499,granite-code-latest-0_45120,439,granite-code-latest-0,0,import pandas as pd\n\n# リストを作る\ndef make_list...,Input string is valid according to the grammar.\n,NameError,failed: name 'f_45120' is not defined,False
501,granite-code-latest-0_38598,414,granite-code-latest-0,0,import pandas as pd\nimport numpy as np\n\ndef...,Input string is valid according to the grammar.\n,AssertionError,failed: 'condition',False
502,granite-code-latest-0_51160,340,granite-code-latest-0,0,"def f_51160(f, n):\n result = 0\n for i ...",Input string is valid according to the grammar.\n,TypeError,failed: 'Pow' object is not callable,False
503,granite-code-latest-0_33677,380,granite-code-latest-0,0,"def f_33677(f):\n \n # x^0, x^1, ..., x^...",Input string is valid according to the grammar.\n,NameError,failed: name 'n' is not defined,False


### False Positive

In [97]:
fp = grammar_summary[grammar_summary["execute"]==1]
fp_grammar = fp[fp["grammar_str"].str.contains("invalid")]
fp_grammar.groupby('model')['execute_message'].count()

model
claude-3-opus-20240229-0    22
gpt-3.5-turbo-0125-0        20
gpt-4-0613-0                21
granite-code-latest-0       28
Name: execute_message, dtype: int64

In [102]:
sampled_fp_grammar = fp_grammar.groupby('model', group_keys=False).apply(lambda x: x if len(x) <= 25 else x.sample(n=25))
sampled_fp_grammar.columns

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_52428/1160475607.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_fp_grammar = fp_grammar.groupby('model', group_keys=False).apply(lambda x: x if len(x) <= 25 else x.sample(n=25))


Index(['completion_id', 'cid', 'model', 'execute', 'lines', 'grammar_str',
       'error_type', 'execute_message', 'grammar'],
      dtype='object')

In [103]:
sampled_fp_grammar[['completion_id', 'cid', 'model', 'execute', 'lines', 'grammar_str']].to_csv("grammar_fp_odex.csv",index=False)

In [101]:
len(sampled_fn_grammar)

100

### Overlap Odex
#### Caught

In [17]:
grammar_summary

,completion_id,cid,model,execute,lines,grammar_str,error_type,execute_message,grammar,grammar_valid
0,claude-3-opus-20240229-0_2195340,8,claude-3-opus-20240229-0,1,import re\n\ndef f_2195340():\n \n strin...,Input string is valid according to the grammar.\n,passed,passed,True,True
1,claude-3-opus-20240229-0_29784889,5,claude-3-opus-20240229-0,1,import warnings\n\ndef f_29784889():\n \n ...,Input string is valid according to the grammar.\n,passed,passed,True,True
2,claude-3-opus-20240229-0_16344756,15,claude-3-opus-20240229-0,1,def f_16344756(app):\n \n app.debug = Tr...,Input string is valid according to the grammar.\n,passed,passed,True,True
3,claude-3-opus-20240229-0_22676,12,claude-3-opus-20240229-0,1,import requests\n\ndef f_22676(url):\n \n ...,Input string is valid according to the grammar.\n,passed,passed,True,True
4,claude-3-opus-20240229-0_698223,6,claude-3-opus-20240229-0,1,from datetime import datetime\n\ndef f_698223(...,Input string is valid according to the grammar.\n,AssertionError,failed:,False,True
...,...,...,...,...,...,...,...,...,...,...
500,granite-code-latest-0_191740,504,granite-code-latest-0,1,"def f_191740(df, col):\n \n return df[co...",Input string is valid according to the grammar.\n,passed,passed,True,True
501,granite-code-latest-0_38598,414,granite-code-latest-0,0,import pandas as pd\nimport numpy as np\n\ndef...,Input string is valid according to the grammar.\n,AssertionError,failed: 'condition',False,True
502,granite-code-latest-0_51160,340,granite-code-latest-0,0,"def f_51160(f, n):\n result = 0\n for i ...",Input string is valid according to the grammar.\n,TypeError,failed: 'Pow' object is not callable,False,True
503,granite-code-latest-0_33677,380,granite-code-latest-0,0,"def f_33677(f):\n \n # x^0, x^1, ..., x^...",Input string is valid according to the grammar.\n,NameError,failed: name 'n' is not defined,False,True


In [39]:
grammar_summary['id'] = grammar_summary['completion_id'].apply(lambda x: x[x.find('_')+1:])
g_v = grammar_summary[['model','id','cid','execute','grammar_valid','lines','grammar_str']]
#Analysis
g_v = g_v[g_v['grammar_valid']==False]
g_v = g_v[g_v['execute']==0]
g_v['id']= g_v['id'].astype(int)

In [40]:
g_v

,model,id,cid,execute,grammar_valid,lines,grammar_str
11,claude-3-opus-20240229-0,26153795,3,0,False,from datetime import datetime\n\ndef f_2615379...,Input string is invalid according to the gramm...
20,claude-3-opus-20240229-0,20573459,24,0,False,"from datetime import datetime, timedelta\n\nde...",Input string is invalid according to the gramm...
31,claude-3-opus-20240229-0,19068269,40,0,False,from datetime import datetime\n\ndef f_1906826...,Input string is invalid according to the gramm...
128,claude-3-opus-20240229-0,4530069,142,0,False,from datetime import datetime\n\ndef f_4530069...,Input string is invalid according to the gramm...
211,claude-3-opus-20240229-0,10607688,226,0,False,from datetime import datetime\n\ndef f_1060768...,Input string is invalid according to the gramm...
...,...,...,...,...,...,...,...
424,granite-code-latest-0,18684076,208,0,False,import pandas as pd\nfrom datetime import time...,Input string is invalid according to the gramm...
432,granite-code-latest-0,34981,447,0,False,"import cv2 as cv\nimg = cv.imread('file.png', ...",Input string is invalid according to the gramm...
435,granite-code-latest-0,40646,450,0,False,"def f_40646(html):\n \n import re,html.p...",Input string is invalid according to the gramm...
457,granite-code-latest-0,44385,472,0,False,def f_44385(command):\n \n process = sub...,Input string is invalid according to the gramm...


In [59]:
static_summary['id'] = static_summary['completion_id'].apply(lambda x: x[x.find('_')+1:])
static_summary['id'] = static_summary['id'].apply(lambda x: x[:x.find('.')])
static_summary['model'] = static_summary['completion_id'].apply(lambda x: x[:x.find('_')])

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_6764/440001102.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  static_summary['id'] = static_summary['completion_id'].apply(lambda x: x[x.find('_')+1:])
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_6764/440001102.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  static_summary['id'] = static_summary['id'].apply(lambda x: x[:x.find('.')])
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_6764/440001102.py:3: SettingWithCo

In [21]:
for l in g_v[(g_v['id']==26153795)]['lines']:
    print(l)

from datetime import datetime

def f_26153795(when):
    
    date_obj = datetime.strptime(when, "%Y-%m-%d %H:%M:%S")
    return date_obj.strftime("%Y-%m-%d")




In [66]:
s_v = static_summary[['model','id','cid','intersect','lines','execute','error_type','execute_message','pyright_str','mypy_str']]
s_v = s_v[s_v["execute"]==0]
s_v = s_v[s_v["intersect"]==0.0]
s_v['id']= s_v['id'].astype(int)

In [67]:
s_v

,model,id,cid,intersect,lines,execute,error_type,execute_message,pyright_str,mypy_str
3,claude-3-opus-20240229-0,10607688,226,0.0,\nfrom datetime import datetime\n\ndef f_10607...,0.0,AttributeError,failed: module 'datetime' has no attribute 'now',"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at..."
12,claude-3-opus-20240229-0,11582,406,0.0,"\ndef f_11582():\n """"""ファイル`C:\Users\Documen...",0.0,SyntaxError,failed: (unicode error) 'unicodeescape' codec ...,"line 11, column 22 : ""file_path"" is possibly u...","line -1, column 0 : )\n"
99,claude-3-opus-20240229-0,19068269,40,0.0,\nfrom datetime import datetime\n\ndef f_19068...,0.0,AttributeError,failed: module 'datetime' has no attribute 'st...,"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at..."
106,claude-3-opus-20240229-0,19685,466,0.0,\nfrom datetime import datetime\n\ndef f_19685...,0.0,TypeError,"failed: strptime() argument 1 must be str, not...","line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at..."
107,claude-3-opus-20240229-0,19685,467,0.0,\nfrom datetime import datetime\n\ndef f_19685...,0.0,AttributeError,failed: module 'datetime' has no attribute 'fr...,"line 1, column 21 : ""datetime"" is unknown impo...","line 1, column 0 : Module ""datetime"" has no at..."
...,...,...,...,...,...,...,...,...,...,...
1990,granite-code-latest-0,7961363,98,0.0,\ndef f_7961363(lst):\n return sorted(list(...,0.0,SyntaxError,failed: Missing parentheses in call to 'print'...,"line 5, column 10 : Statements must be separat...","line -1, column 0 : )\n"
1991,granite-code-latest-0,804995,168,0.0,"\nimport shlex, subprocess\nfrom .log import *...",0.0,KeyError,"failed: ""'__name__' not in globals""","line 2, column 5 : Import "".log"" could not be ...","line 2, column 0 : Cannot find implementation ..."
1998,granite-code-latest-0,8519922,79,0.0,"\nimport urllib.request, urllib.parse, urllib....",0.0,KeyError,"failed: ""'__name__' not in globals""","line 2, column 5 : Import "".models"" could not ...","line 2, column 0 : Cannot find implementation ..."
2001,granite-code-latest-0,8740353,221,0.0,"\nimport urlparse, urllib, base64\nimport simp...",0.0,ModuleNotFoundError,failed: No module named 'urlparse',"line 1, column 7 : Import ""urlparse"" could not...","line 1, column 0 : Cannot find implementation ..."


In [70]:
intersection = pd.merge(s_v[['model','id','cid','error_type','execute_message','lines','pyright_str','mypy_str']], g_v[['model','id','cid','grammar_str']], how='left', on=['model','id','cid'])

In [73]:
intersection.columns

Index(['model', 'id', 'cid', 'error_type', 'execute_message', 'lines',
       'pyright_str', 'mypy_str', 'grammar_str'],
      dtype='object')

In [72]:
intersection.to_csv('odex_detected_reasons.csv',index=False)

In [49]:
only_in_df1 = pd.concat([g_v[['model','id','cid']], s_v[['model','id','cid']]]).drop_duplicates(keep=False)

In [50]:
only_in_df1

,model,id,cid
11,claude-3-opus-20240229-0,26153795,3
20,claude-3-opus-20240229-0,20573459,24
31,claude-3-opus-20240229-0,19068269,40
128,claude-3-opus-20240229-0,4530069,142
211,claude-3-opus-20240229-0,10607688,226
...,...,...,...
2015,granite-code-latest-0,9760588,84
2016,granite-code-latest-0,9760588,85
2017,granite-code-latest-0,9775297,178
2019,granite-code-latest-0,98725,498


In [51]:
intersection = pd.merge(g_v[['model','cid']], s_v[['model','cid']], how='inner', on=['model','cid'])
intersection

,model,cid
0,claude-3-opus-20240229-0,273
1,claude-3-opus-20240229-0,397
2,claude-3-opus-20240229-0,404
3,claude-3-opus-20240229-0,475
4,granite-code-latest-0,46
5,granite-code-latest-0,0
6,granite-code-latest-0,47
7,granite-code-latest-0,142
8,granite-code-latest-0,183
9,granite-code-latest-0,139


In [12]:
ds_detected = pd.read_csv('ds_detected_reasons.csv')

In [14]:
ds_detected.groupby(['grammar_capable'])['lines'].count()

grammar_capable
No     70
Yes    22
Name: lines, dtype: int64

In [20]:
134-92

42

In [15]:
ds_detected.groupby(['static_capable'])['lines'].count()

static_capable
No     25
Yes    67
Name: lines, dtype: int64

In [21]:
42+67

109

In [17]:
odex_detected = pd.read_csv('odex_detected_reasons.csv')

In [18]:
odex_detected.groupby(['grammar_capable'])['lines'].count()

grammar_capable
No      4
Yes    26
Name: lines, dtype: int64

In [19]:
odex_detected.groupby(['static_capable'])['lines'].count()

static_capable
No      1
Yes    29
Name: lines, dtype: int64